<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L3/03_asl_cnn_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="text-align:center">

# Applied Deep Learning
### Convolutional Neural Networks for ASL Image Classification

</div>

>
> **Dataset:** Sign Language MNIST (`datamunge/sign-language-mnist`), distributed on Kaggle under **CC0: Public Domain**.

# 3. Convolutional Neural Networks

In the previous section, we built and trained a simple model to classify ASL images. The model was able to learn how to correctly classify the training dataset with very high accuracy, but it did not perform nearly as well on the validation dataset. This behavior of not generalizing well to non-training data is called [overfitting](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html).

In this section, we introduce a popular model architecture called a **convolutional neural network (CNN)**. CNNs are especially useful for image data because they preserve and learn from spatial relationships between nearby pixels.

## 3.1 Objectives

* Prep data specifically for a CNN
* Create a more sophisticated CNN model, understanding a greater variety of model layers
* Train a CNN model and observe its performance

In [1]:
# Download the public Sign Language MNIST dataset from Kaggle.
# The dataset is CC0: Public Domain.

import sys
import subprocess
import importlib.util
from pathlib import Path

if importlib.util.find_spec("kagglehub") is None:
    print("Installing kagglehub...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

import kagglehub

dataset_path = Path(kagglehub.dataset_download("datamunge/sign-language-mnist"))

train_matches = list(dataset_path.rglob("sign_mnist_train.csv"))
test_matches = list(dataset_path.rglob("sign_mnist_test.csv"))

if not train_matches or not test_matches:
    raise FileNotFoundError("Could not locate the Sign Language MNIST CSV files.")

TRAIN_CSV = train_matches[0]
VALID_CSV = test_matches[0]

print("Training CSV  :", TRAIN_CSV)
print("Validation CSV:", VALID_CSV)

Using Colab cache for faster access to the 'sign-language-mnist' dataset.
Training CSV  : /kaggle/input/sign-language-mnist/sign_mnist_train.csv
Validation CSV: /kaggle/input/sign-language-mnist/sign_mnist_test.csv


In [3]:
import torch.nn as nn
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
Selected device: cuda
GPU: Tesla T4


## 3.2 Loading and Preparing the Data

### 3.2.1 Preparing Images

Let's begin by loading our DataFrames like we did in the previous lab:

In [4]:
train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

# The public Kaggle dataset uses original alphabet-position labels.
# J and Z are absent, so the raw labels are not contiguous.
# CrossEntropyLoss expects class indices 0...(C-1), so remap them.
original_labels = sorted(train_df["label"].unique().tolist())
label_to_index = {label: idx for idx, label in enumerate(original_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}
index_to_letter = {
    idx: chr(ord("A") + original_label)
    for idx, original_label in index_to_label.items()
}

train_df["label"] = train_df["label"].map(label_to_index)
valid_df["label"] = valid_df["label"].map(label_to_index)

print("Training shape  :", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Classes         :", len(original_labels))
print("Letters         :", [index_to_letter[i] for i in range(len(original_labels))])

Training shape  : (27455, 785)
Validation shape: (7172, 785)
Classes         : 24
Letters         : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


This ASL data is already flattened.

In [5]:
sample_df = train_df.head().copy()  # Grab the top 5 rows
sample_df.pop('label')
sample_x = sample_df.values
sample_x

array([[107, 118, 127, ..., 204, 203, 202],
       [155, 157, 156, ..., 103, 135, 149],
       [187, 188, 188, ..., 195, 194, 195],
       [211, 211, 212, ..., 222, 229, 163],
       [164, 167, 170, ..., 163, 164, 179]])

In [6]:
sample_x.shape

(5, 784)

In this format, we don't have all the information about which pixels are near each other. Because of this, we can't apply convolutions that will detect features. Let's [reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) our dataset so that they are in a 28x28 pixel format. This will allow our convolutions to associate groups of pixels and detect important features.

Note that for the first convolutional layer of our model, we need to have not only the height and width of the image, but also the number of [color channels](https://www.photoshopessentials.com/essentials/rgb/). Our images are grayscale, so we'll just have 1 channel.

That means that we need to convert the current shape `(5, 784)` to `(5, 1, 28, 28)`. With [NumPy](https://numpy.org/doc/stable/index.html) arrays, we can pass a `-1` for any dimension we wish to remain the same.

In [7]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

sample_x = sample_x.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
sample_x.shape

(5, 1, 28, 28)

### 3.2.2 Create a Dataset

Let's add the steps above into our `MyDataset` class.

#### Exercise

There are 4 `FIXME`s in the class definition below. Can you replace them with the correct values?

In [ ]:
class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()  # Some operations below are in-place
        y_df = x_df.pop(FIXME)
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, FIXME, FIXME, FIXME)
        self.xs = torch.tensor(x_df).float()
        self.ys = torch.tensor(y_df.values).long()

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

#### Solution

Click the `...` below for the solution.

In [22]:
#@title

# SOLUTION
class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()  # Some operations below are in-place
        y_df = x_df.pop('label')
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, IMG_CHS, IMG_WIDTH, IMG_HEIGHT)
        self.xs = torch.tensor(x_df).float()
        self.ys = torch.tensor(y_df.values).long()

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

### 3.2.3 Create a DataLoader

Next, let's create the DataLoader from the Dataset

#### Exercise

One of these function calls is missing the `shuffle=True` argument. Can you remember which one it is and add it back in?

In [9]:
BATCH_SIZE = 32

train_data = MyDataset(train_df)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
valid_N = len(valid_loader.dataset)

#### Solution

Click the `...` below for the solution.

In [10]:
# SOLUTION
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

Let's grab a batch from the DataLoader to make sure it works.

In [11]:
batch = next(iter(train_loader))
batch

[tensor([[[[0.5333, 0.5608, 0.5922,  ..., 0.8000, 0.8039, 0.8078],
           [0.5373, 0.5647, 0.5961,  ..., 0.8078, 0.8118, 0.8118],
           [0.5490, 0.5725, 0.6039,  ..., 0.8157, 0.8157, 0.8157],
           ...,
           [0.4039, 0.4078, 0.4078,  ..., 0.4275, 0.5647, 0.6314],
           [0.3961, 0.4000, 0.4000,  ..., 0.4196, 0.5255, 0.6471],
           [0.3922, 0.4000, 0.4000,  ..., 0.4196, 0.4824, 0.6627]]],
 
 
         [[[0.4941, 0.4941, 0.4941,  ..., 0.4980, 0.4941, 0.4980],
           [0.5020, 0.5020, 0.5020,  ..., 0.5098, 0.5059, 0.5059],
           [0.5137, 0.5137, 0.5137,  ..., 0.5098, 0.5098, 0.5137],
           ...,
           [0.6039, 0.5686, 0.6275,  ..., 0.6627, 0.6588, 0.6588],
           [0.5725, 0.6000, 0.6118,  ..., 0.6627, 0.6588, 0.6588],
           [0.5647, 0.6078, 0.6039,  ..., 0.6706, 0.6667, 0.6627]]],
 
 
         [[[0.7294, 0.7333, 0.7373,  ..., 0.7216, 0.7137, 0.7137],
           [0.7412, 0.7490, 0.7529,  ..., 0.7412, 0.7294, 0.7255],
           [0.7569

It looks different, but let's check the `shape`s to be sure.

In [12]:
batch[0].shape

torch.Size([32, 1, 28, 28])

In [13]:
batch[1].shape

torch.Size([32])

## 3.3 Creating a Convolutional Model

These days, many data scientists begin by studying model architectures that have worked well on similar problems. For image classification, convolutional neural networks provide a strong starting point.

For this practical, we will use the following CNN architecture:

```text
# INPUT
1 × 28 × 28

# CONVOLUTION BLOCK 1
    ↓ Conv2d(1 → 25) + BatchNorm + ReLU
25 × 28 × 28
    ↓ MaxPool
25 × 14 × 14

# CONVOLUTION BLOCK 2
    ↓ Conv2d(25 → 50) + BatchNorm + ReLU + Dropout
50 × 14 × 14
    ↓ MaxPool
50 × 7 × 7

# CONVOLUTION BLOCK 3
    ↓ Conv2d(50 → 75) + BatchNorm + ReLU
75 × 7 × 7
    ↓ MaxPool
75 × 3 × 3

# FLATTEN
    ↓ Flatten
675

# FULLY CONNECTED / DENSE BLOCK
    ↓ Linear(675 → 512) + Dropout + ReLU
512

# OUTPUT LAYER
    ↓ Linear(512 → 24)
24 class logits
```

We covered many of these layer types in the lecture. We will go over them here and link to the official PyTorch documentation.

In [14]:
n_classes = 24
kernel_size = 3
flattened_img_size = 75 * 3 * 3

model = nn.Sequential(
    # First convolution
    nn.Conv2d(IMG_CHS, 25, kernel_size, stride=1, padding=1),  # 25 x 28 x 28
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 25 x 14 x 14
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # 50 x 14 x 14
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # 50 x 7 x 7
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # 75 x 7 x 7
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 75 x 3 x 3
    # Flatten to Dense
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, n_classes)
)

### 3.3.1 [Conv2D](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

<img src="https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/iti121/images/conv2d.png" width=300 />

These are our 2D convolutional layers. Small kernels move across the input image and learn features that are useful for classification. Earlier convolutional layers often respond to simpler visual patterns such as edges and lines, while deeper layers can combine them into more complex features.

Let's look at our first Conv2D layer:

```python
nn.Conv2d(IMG_CHS, 25, kernel_size, stride=1, padding=1)
```

- `IMG_CHS` is the number of input channels. Our images are grayscale, so this is 1.
- `25` is the number of filters (output channels) that will be learned.
- `kernel_size = 3` gives us 3 × 3 filters.
- `stride=1` means the filter moves one pixel at a time.
- `padding=1` preserves the spatial height and width for a 3 × 3 kernel.

Official documentation: [torch.nn.Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

### 3.3.2 [BatchNormalization](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)

Like normalizing our inputs, batch normalization scales the values in the hidden layers to improve training. [Read more about it in detail here](https://blog.paperspace.com/busting-the-myths-about-batch-normalization/).

There is a debate on best where to put the batch normalization layer. [This Stack Overflow post](https://stackoverflow.com/questions/39691902/ordering-of-batch-normalization-and-dropout) compiles many perspectives.

### 3.3.3 [MaxPool2D](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)

<img src="https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/iti121/images/maxpool2d.png" width=300 />

Max pooling takes an image and essentially shrinks it to a lower resolution. It does this to help the model be robust to translation (objects moving side to side), and also makes our model faster.
Max pooling reduces the spatial resolution of the feature maps. It keeps the strongest activation in each pooling window, making the representation smaller and reducing computation.

For example:

```text
28 × 28
   ↓ MaxPool2d(2)
14 × 14
```

Pooling can also make the representation less sensitive to small translations in the input.

Official documentation: [torch.nn.MaxPool2d](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)

### 3.3.4 [Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

Dropout is a technique for reducing overfitting. During training, it randomly sets a fraction of activations to zero. This discourages the network from relying too heavily on any one activation pathway.

<img src="https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/iti121/images/dropout.png" width=360 />

For example:

```python
nn.Dropout(0.2)
```

means that approximately 20% of the activations are dropped during each training pass.

Dropout is active during `model.train()` and disabled during `model.eval()`.

Official documentation: [torch.nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

### 3.3.5 [Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)

Flatten takes the output of one layer which is multidimensional, and flattens it into a one-dimensional array. The output is called a feature vector and will be connected to the final classification layer.

### 3.3.6 [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)

We have seen dense linear layers before in our earlier models. Our first dense layer (512 units) takes the feature vector as input and learns which features will contribute to a particular classification. The second dense layer (24 units) is the final classification layer that outputs our prediction.

## 3.4 Summarizing the Model

This may feel like a lot of information, but don't worry. It's not critical that to understand everything right now in order to effectively train convolutional models. Most importantly we know that they can help with extracting useful information from images, and can be used in classification tasks.

In [15]:
model = model.to(device)

# Optional performance optimisation for supported environments:
# model = torch.compile(model)

model

Sequential(
  (0): Conv2d(1, 25, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(25, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (4): Conv2d(25, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Dropout(p=0.2, inplace=False)
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Conv2d(50, 75, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (10): BatchNorm2d(75, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): ReLU()
  (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (13): Flatten(start_dim=1, end_dim=-1)
  (14): Linear(in_features=675, out_features=512, bias=True)
  (15): Dropout(p=0.3, inplace=False)
  (16): ReLU()
  (17): Linear(in_features

Since the problem we are trying to solve is still the same (classifying ASL images), we will continue to use the same `CrossEntropyLoss` and accuracy metric.

`CrossEntropyLoss` expects raw logits from the final layer, so we do **not** add a `Softmax` layer to the model.

In [16]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters())

In [17]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

### 3.5 Training the Model

Despite the very different model architecture, the training looks exactly the same.

#### Exercise

These are the same `train` and `validate` functions as before, but they have been mixed up. Can you correctly name each function and replace the `FIXME`s?

One of them should have `model.train` and the other should have `model.eval`.

In [ ]:
def FIXME():
    loss = 0
    accuracy = 0

    model.FIXME()
    with torch.no_grad():
        for x, y in FIXME:
            x = x.to(device)
            y = y.to(device)

            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)

    print('FIXME - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [ ]:
def FIXME():
    loss = 0
    accuracy = 0

    model.FIXME()
    for x, y in FIXME:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)

    print('FIXME - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

#### Solution

Click the two `...`s below for the solution.

In [23]:
#@title

# SOLUTION
def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)

    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [24]:
#@title

# SOLUTION
def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)

    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [20]:
epochs = 20

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

Epoch: 0
Train - Loss: 252.8726 Accuracy: 0.9131
Valid - Loss: 24.3957 Accuracy: 0.9682
Epoch: 1
Train - Loss: 15.9596 Accuracy: 0.9954
Valid - Loss: 30.4961 Accuracy: 0.9483
Epoch: 2
Train - Loss: 12.4252 Accuracy: 0.9959
Valid - Loss: 18.5072 Accuracy: 0.9709
Epoch: 3
Train - Loss: 14.9417 Accuracy: 0.9944
Valid - Loss: 54.2109 Accuracy: 0.9406
Epoch: 4
Train - Loss: 5.2288 Accuracy: 0.9981
Valid - Loss: 12.9710 Accuracy: 0.9759
Epoch: 5
Train - Loss: 4.9953 Accuracy: 0.9984
Valid - Loss: 45.4654 Accuracy: 0.9346
Epoch: 6
Train - Loss: 6.0881 Accuracy: 0.9980
Valid - Loss: 19.8023 Accuracy: 0.9749
Epoch: 7
Train - Loss: 9.4221 Accuracy: 0.9973
Valid - Loss: 22.1598 Accuracy: 0.9742
Epoch: 8
Train - Loss: 8.6609 Accuracy: 0.9972
Valid - Loss: 28.5610 Accuracy: 0.9601
Epoch: 9
Train - Loss: 1.3427 Accuracy: 0.9997
Valid - Loss: 19.0068 Accuracy: 0.9743
Epoch: 10
Train - Loss: 9.3427 Accuracy: 0.9967
Valid - Loss: 19.6375 Accuracy: 0.9782
Epoch: 11
Train - Loss: 4.4150 Accuracy: 0.9986


### 3.5.1 Discussion of Results

The CNN should generally perform better than a simple fully connected model because convolutional layers preserve and learn from spatial structure in the images.

Compare the training and validation accuracy from your own run:

- If both are high and close together, the model is generalising reasonably well.
- If training accuracy is much higher than validation accuracy, the model is still overfitting.
- If validation accuracy fluctuates substantially between epochs, the model may still benefit from additional regularisation or more varied training data.

In the next practical, we will introduce **data augmentation** to improve generalisation.

## 3.6 Summary

In this section, we utilized several new kinds of layers to implement a CNN, which performed better than the more simple model used in the last section. Hopefully the overall process of creating and training a model with prepared data is starting to become even more familiar.

### 3.6.1 Next

In the last several sections you have focused on the creation and training of models. To further improve generalisation, we will now turn our attention to **data augmentation** — techniques that create useful variation in the training data.

---
### Dataset Attribution

**Sign Language MNIST** — Kaggle dataset by `datamunge`  
License: **CC0: Public Domain**  
Dataset page: `https://www.kaggle.com/datasets/datamunge/sign-language-mnist`
